# dataloader-batching — faded example 3: Compute the dataset mean by streaming batches

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `dataloader-batching`. The last cell reports your progress on the `PyTorch: DataLoader batching` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: DataLoader batching` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`dataloader-batching`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "dataloader-batching"
DD_SUBTOPIC = "PyTorch: DataLoader batching"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A `DataLoader` lets you process a dataset in batches without loading everything at once. To compute the exact global mean of a feature tensor, accumulate the per-batch sums and the per-batch counts, then divide total-sum by total-count. This weighted accumulation gives the same result as `x.mean(0)` even when the last batch is partial.

## Faded exercise 3

### Faded — streaming mean over a DataLoader

Implement `streaming_mean(x, B)`. Wrap the 2-D feature tensor `x` of shape `(N, D)` in a `TensorDataset`, iterate a `DataLoader(batch_size=B, shuffle=False)`, and accumulate a running sum and a running count so you can return the column-wise mean of shape `(D,)`.

The accumulation loop and the final division are written. **Complete the line that updates the running sum with the current batch's contribution.**

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader


def streaming_mean(x, B):
    ds = TensorDataset(x)
    loader = DataLoader(ds, batch_size=B, shuffle=False)
    D = x.shape[1]
    total_sum = t.zeros(D)
    total_count = 0
    for (xb,) in loader:
        raise NotImplementedError()  # TODO: fill in this step — read the prompt cell above
        total_count += xb.shape[0]
    return total_sum / total_count


def _test():
    t.manual_seed(0)
    for N, D, B in [(23, 4, 5), (16, 3, 4), (10, 2, 7)]:
        x = t.randn(N, D)
        got = streaming_mean(x, B)
        expected = x.mean(0)
        assert got.shape == (D,), (N, D, B, got.shape)
        assert t.allclose(got, expected, atol=1e-5), (N, D, B, got, expected)


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
from torch.utils.data import TensorDataset, DataLoader


def streaming_mean(x, B):
    ds = TensorDataset(x)
    loader = DataLoader(ds, batch_size=B, shuffle=False)
    D = x.shape[1]
    total_sum = t.zeros(D)
    total_count = 0
    for (xb,) in loader:
        total_sum = total_sum + xb.sum(0)
        total_count += xb.shape[0]
    return total_sum / total_count
```
</details>